In [1]:
# @title 1. 환경 설정 및 라이브러리 설치
import os
import sys

try:
    from google.colab import drive, files, userdata
    IN_COLAB = True
except ImportError:
    drive = files = userdata = None
    IN_COLAB = False

if IN_COLAB:
    get_ipython().run_line_magic(
        "pip",
        "install -q -U google-genai openai pydantic openpyxl xlrd ipywidgets",
    )
    get_ipython().system(
        "DEBIAN_FRONTEND=noninteractive apt-get install -yqq fonts-nanum >/dev/null"
    )

import base64
import datetime as dt
import io
import json
import mimetypes
import tempfile
import traceback
import uuid
from pathlib import Path
from typing import Any, Literal

import ipywidgets as widgets
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, Image, Markdown, clear_output, display
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

from google import genai
from google.genai import types
from openai import OpenAI

FONT_PATH = "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"
if Path(FONT_PATH).exists():
    fm.fontManager.addfont(FONT_PATH)
    FONT_NAME = fm.FontProperties(fname=FONT_PATH).get_name()
    plt.rc("font", family=FONT_NAME)
    plt.rcParams["axes.unicode_minus"] = False
    sns.set_theme(style="whitegrid", font=FONT_NAME, rc={"axes.unicode_minus": False})
else:
    sns.set_theme(style="whitegrid")

sns.set_palette("viridis")
print(f"환경 설정 완료 · 실행 환경: {'Google Colab' if IN_COLAB else '로컬 검증'}")


환경 설정 완료 · 실행 환경: 로컬 검증


In [2]:
# @title 2. Colab Secrets API 인증
SECRET_NAMES = ("OPENAI_API_KEY", "GEMINI_API_KEY", "GOOGLE_API_KEY", "exhibition")


def read_secret(name):
    """Secret 값을 반환하되 값 자체는 출력하거나 기록하지 않습니다."""
    if IN_COLAB:
        try:
            return userdata.get(name) or None
        except Exception:
            return None
    return os.environ.get(name) or None


def select_gemini_key(values):
    return values.get("GEMINI_API_KEY") or values.get("GOOGLE_API_KEY") or values.get("exhibition")


SECRET_VALUES = {name: read_secret(name) for name in SECRET_NAMES}
API_KEYS = {
    "OpenAI": SECRET_VALUES.get("OPENAI_API_KEY"),
    "Gemini": select_gemini_key(SECRET_VALUES),
}
DEFAULT_API_PROVIDER = "OpenAI"

# 원본 Secret 사전은 더 이상 필요하지 않으며 대화·로그에 저장하지 않습니다.
del SECRET_VALUES


def safe_error_message(error):
    text = str(error)
    for secret in API_KEYS.values():
        if secret:
            text = text.replace(secret, "[REDACTED]")
    return text


available = [provider for provider, value in API_KEYS.items() if value]
if available:
    print("API 인증 준비 완료:", ", ".join(available))
else:
    print("API 키 없음 · 데이터 적재, 전처리, 구조화 시각화는 계속 사용할 수 있습니다.")


API 키 없음 · 데이터 적재, 전처리, 구조화 시각화는 계속 사용할 수 있습니다.


In [3]:
# @title 3. 기본 테스트 파일 및 분석 데이터 선택
DEFAULT_DRIVE_DIRECTORY = Path("/content/drive/MyDrive/Exhibition_Data_Analysis/default_test_file")
LOCAL_TEST_DIRECTORY = Path(tempfile.gettempdir()) / "exhibition_data_analysis_notebook"
DEFAULT_DIRECTORY = DEFAULT_DRIVE_DIRECTORY if IN_COLAB else LOCAL_TEST_DIRECTORY
DEFAULT_MANIFEST = "manifest.json"


def ensure_drive_ready():
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=False)
    DEFAULT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    return DEFAULT_DIRECTORY


def upload_items(value):
    if not value:
        return []
    raw_items = list(value.values()) if isinstance(value, dict) else list(value)
    result = []
    for item in raw_items:
        if isinstance(item, dict):
            result.append({
                "name": item.get("name", "upload"),
                "type": item.get("type", "application/octet-stream"),
                "content": bytes(item.get("content", b"")),
            })
        else:
            result.append({
                "name": getattr(item, "name", "upload"),
                "type": getattr(item, "type", "application/octet-stream"),
                "content": bytes(getattr(item, "content", b"")),
            })
    return result


def read_csv_bytes(raw):
    last_error = None
    for encoding in ("utf-8-sig", "cp949", "utf-8"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error or ValueError("CSV 인코딩을 확인할 수 없습니다.")


def read_table_bytes(filename, raw):
    suffix = Path(filename).suffix.lower()
    if suffix == ".csv":
        frame = read_csv_bytes(raw)
    elif suffix in {".xlsx", ".xls"}:
        frame = pd.read_excel(io.BytesIO(raw))
    else:
        raise ValueError("CSV, XLSX 또는 XLS 파일만 사용할 수 있습니다.")
    if frame.empty and len(frame.columns) == 0:
        raise ValueError("컬럼이 없는 빈 파일입니다.")
    return frame


def dataset_record(filename, raw, origin):
    frame = read_table_bytes(filename, raw)
    return {
        "id": f"{origin}:{uuid.uuid4().hex}",
        "filename": Path(filename).name,
        "raw": bytes(raw),
        "frame": frame,
        "origin": origin,
        "rows": int(len(frame)),
        "columns": int(frame.shape[1]),
    }


SUPPORTED_DATA_EXTENSIONS = {".csv", ".xlsx", ".xls"}


def latest_supported_upload(value):
    """Return the most recently selected supported file from Colab's accumulated queue."""
    items = upload_items(value)
    if not items:
        raise ValueError("업로드할 파일을 선택하세요.")
    valid = [
        item for item in items
        if Path(str(item.get("name", ""))).suffix.lower() in SUPPORTED_DATA_EXTENSIONS
    ]
    if not valid:
        received = ", ".join(str(item.get("name", "이름 없음")) for item in items)
        raise ValueError(
            "CSV, XLSX 또는 XLS 파일만 사용할 수 있습니다. "
            f"선택된 파일: {received}"
        )
    selected = valid[-1]
    ignored = [
        str(item.get("name", "이름 없음")) for item in items
        if item is not selected
    ]
    return selected, ignored


def clear_upload_widget(widget):
    """Clear FileUpload across ipywidgets 7/8 without affecting the parsed dataset."""
    empty_value = () if isinstance(widget.value, tuple) else {}
    try:
        widget.value = empty_value
    except Exception:
        pass
    if hasattr(widget, "_counter"):
        try:
            widget._counter = 0
        except Exception:
            pass


def default_paths(directory=None):
    base = Path(directory or DEFAULT_DIRECTORY)
    return base, base / DEFAULT_MANIFEST


def save_default_dataset(record, directory=None):
    base, manifest_path = default_paths(directory)
    base.mkdir(parents=True, exist_ok=True)
    suffix = Path(record["filename"]).suffix.lower()
    stored_name = f"default_test_file{suffix}"
    for old in base.glob("default_test_file.*"):
        if old.name != stored_name:
            old.unlink(missing_ok=True)
    (base / stored_name).write_bytes(record["raw"])
    manifest = {
        "filename": record["filename"],
        "stored_name": stored_name,
        "rows": record["rows"],
        "columns": record["columns"],
        "saved_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return manifest


def load_default_dataset(directory=None):
    base, manifest_path = default_paths(directory)
    if not manifest_path.exists():
        return None
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        data_path = base / Path(manifest["stored_name"]).name
        raw = data_path.read_bytes()
        return dataset_record(manifest["filename"], raw, "default")
    except Exception as exc:
        print(f"[경고] 저장된 기본 테스트 파일을 불러오지 못했습니다: {exc}")
        return None


def delete_default_dataset(directory=None):
    base, manifest_path = default_paths(directory)
    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            (base / Path(manifest.get("stored_name", "")).name).unlink(missing_ok=True)
        except Exception:
            pass
    manifest_path.unlink(missing_ok=True)


def dataset_description(record):
    if not record:
        return "없음"
    return f"{record['filename']} · {record['rows']:,}행 × {record['columns']:,}열"


ensure_drive_ready()
default_dataset = load_default_dataset()
uploaded_dataset = None
df = None
df_clean = None
source_filename = None
analysis_source = None
preprocessing_history = []
chart_history = []
current_chart_spec = None
current_chart_statistics = None


def reset_downstream_state():
    global preprocessing_history, chart_history, current_chart_spec, current_chart_statistics
    preprocessing_history = []
    chart_history = []
    current_chart_spec = None
    current_chart_statistics = None


def set_analysis_dataset(record, source):
    global df, df_clean, source_filename, analysis_source
    if record is None:
        raise ValueError("선택한 데이터가 없습니다.")
    df = record["frame"].copy(deep=True)
    df_clean = df.copy(deep=True)
    source_filename = record["filename"]
    analysis_source = source
    reset_downstream_state()
    if "refresh_analysis_widgets" in globals():
        refresh_analysis_widgets()
    if "refresh_insight_dataset_selector" in globals():
        refresh_insight_dataset_selector()
    refresh_data_status()


def basic_profile(frame):
    if frame is None:
        return pd.DataFrame()
    return pd.DataFrame({
        "변수명": [str(column) for column in frame.columns],
        "데이터 개수": [int(frame[column].notna().sum()) for column in frame.columns],
        "데이터 타입": [str(frame[column].dtype) for column in frame.columns],
        "결측 개수": [int(frame[column].isna().sum()) for column in frame.columns],
    })


data_upload = widgets.FileUpload(
    accept=".csv,.xlsx,.xls", multiple=False, description="새 파일 업로드"
)
btn_register_upload = widgets.Button(description="업로드 파일 등록", button_style="info")
btn_save_default = widgets.Button(description="기본 테스트 파일 저장/교체", button_style="success")
btn_delete_default = widgets.Button(description="기본 테스트 파일 삭제", button_style="danger")
dataset_selector = widgets.RadioButtons(description="분석할 데이터:", options=[])
btn_apply_dataset = widgets.Button(description="선택 데이터 분석", button_style="primary")
btn_restore_default = widgets.Button(description="기본 테스트 파일로 복원")
data_status = widgets.HTML()
data_output = widgets.Output()


def refresh_dataset_options():
    options = []
    if default_dataset:
        options.append((f"저장된 기본 테스트 파일 · {dataset_description(default_dataset)}", "default"))
    if uploaded_dataset:
        options.append((f"이번에 새로 업로드한 파일 · {dataset_description(uploaded_dataset)}", "upload"))
    dataset_selector.options = options
    if options:
        values = [value for _, value in options]
        dataset_selector.value = "default" if "default" in values else values[0]


def refresh_data_status():
    current = "선택되지 않음" if df is None else f"{source_filename} · {len(df):,}행 × {df.shape[1]:,}열"
    data_status.value = (
        f"<b>저장된 기본 파일:</b> {dataset_description(default_dataset)}<br>"
        f"<b>새 업로드 파일:</b> {dataset_description(uploaded_dataset)}<br>"
        f"<b>현재 분석 파일:</b> {current}"
    )
    with data_output:
        clear_output(wait=True)
        if df is not None:
            display(Markdown("### 데이터 기본 현황"))
            display(basic_profile(df))
            display(df.head(5))


def on_register_upload(_):
    global uploaded_dataset
    with data_output:
        clear_output(wait=True)
        try:
            item, ignored = latest_supported_upload(data_upload.value)
            uploaded_dataset = dataset_record(item["name"], item["content"], "upload")
            clear_upload_widget(data_upload)
            refresh_dataset_options()
            refresh_data_status()
            display(Markdown(
                f"**업로드 등록 완료:** {dataset_description(uploaded_dataset)}"
            ))
            if ignored:
                display(Markdown(
                    "이전에 누적된 파일은 무시했습니다: " + ", ".join(ignored)
                ))
        except Exception as exc:
            display(Markdown(f"**업로드 실패:** {exc}"))


def on_save_default(_):
    global default_dataset
    with data_output:
        try:
            if uploaded_dataset is None:
                raise ValueError("먼저 새 파일을 업로드하여 등록하세요.")
            save_default_dataset(uploaded_dataset)
            default_dataset = load_default_dataset()
            refresh_dataset_options()
            refresh_data_status()
            display(Markdown("**기본 테스트 파일을 저장·교체했습니다.**"))
        except Exception as exc:
            display(Markdown(f"**저장 실패:** {exc}"))


def on_delete_default(_):
    global default_dataset
    delete_default_dataset()
    default_dataset = None
    refresh_dataset_options()
    refresh_data_status()


def selected_dataset_record():
    return default_dataset if dataset_selector.value == "default" else uploaded_dataset


def on_apply_dataset(_):
    with data_output:
        try:
            set_analysis_dataset(selected_dataset_record(), dataset_selector.value)
        except Exception as exc:
            display(Markdown(f"**데이터 선택 실패:** {exc}"))


def on_restore_default(_):
    with data_output:
        try:
            if default_dataset is None:
                raise ValueError("저장된 기본 테스트 파일이 없습니다.")
            set_analysis_dataset(default_dataset, "default")
        except Exception as exc:
            display(Markdown(f"**복원 실패:** {exc}"))


btn_register_upload.on_click(on_register_upload)
btn_save_default.on_click(on_save_default)
btn_delete_default.on_click(on_delete_default)
btn_apply_dataset.on_click(on_apply_dataset)
btn_restore_default.on_click(on_restore_default)

refresh_dataset_options()
if default_dataset is not None:
    set_analysis_dataset(default_dataset, "default")
else:
    refresh_data_status()

data_panel = widgets.VBox([
    widgets.HTML("<h3>데이터 파일과 기본 테스트 파일</h3>"),
    widgets.HBox([data_upload, btn_register_upload]),
    widgets.HBox([btn_save_default, btn_delete_default, btn_restore_default]),
    dataset_selector,
    btn_apply_dataset,
    data_status,
    data_output,
])


In [4]:
# @title 4. 전처리 및 구조화 시각화
SUPPORTED_CHARTS = (
    "bar", "line", "pie", "histogram", "scatter", "grouped_bar",
    "stacked_bar", "scatter_bubble", "heatmap", "correlation_heatmap",
)


class ChartSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")

    chart_type: Literal[
        "bar", "line", "pie", "histogram", "scatter", "grouped_bar",
        "stacked_bar", "scatter_bubble", "heatmap", "correlation_heatmap",
    ]
    x: str | None = None
    y: str | None = None
    value_column: str | None = None
    aggregation: Literal["count", "valid_count", "sum", "mean", "ratio"] = "count"
    title: str = ""
    show_values: bool = True
    category_orders: dict[str, list[str]] = Field(default_factory=dict)
    variables: list[str] = Field(default_factory=list)

    @model_validator(mode="after")
    def validate_chart(self):
        if self.chart_type == "correlation_heatmap":
            if len(self.variables) < 2:
                raise ValueError("Correlation Heatmap은 수치형 variables가 2개 이상 필요합니다.")
            return self
        if not self.x:
            raise ValueError("X 변수를 선택하세요.")
        if self.chart_type in {"scatter", "grouped_bar", "stacked_bar", "scatter_bubble", "heatmap"} and not self.y:
            raise ValueError(f"{self.chart_type}에는 Y 변수가 필요합니다.")
        if self.aggregation in {"sum", "mean"} and not self.value_column:
            raise ValueError("합계 또는 평균에는 집계 대상 수치형 변수가 필요합니다.")
        return self


def require_analysis_data():
    if df_clean is None:
        raise ValueError("먼저 분석할 데이터를 선택하세요.")
    return df_clean


def coerce_replacement(series, value):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(pd.Series([value]), errors="raise").iloc[0]
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(value)
    return value


def apply_category_orders(table, spec):
    result = table.copy()
    sort_columns = []
    for column, requested in spec.category_orders.items():
        if column in result and requested:
            observed = result[column].dropna().astype(str).unique().tolist()
            categories = requested + [item for item in observed if item not in requested]
            result[column] = pd.Categorical(result[column].astype(str), categories=categories, ordered=True)
            sort_columns.append(column)
    if sort_columns:
        result = result.sort_values(sort_columns, kind="stable")
        for column in sort_columns:
            result[column] = result[column].astype("object")
    return result.reset_index(drop=True)


def aggregate_statistics(frame, spec, keys):
    working = frame.copy()
    if spec.aggregation == "count":
        table = working.groupby(keys, dropna=False, observed=False).size().reset_index(name="value")
    elif spec.aggregation == "valid_count":
        target = spec.value_column or (spec.y if spec.y not in keys else spec.x)
        table = working.groupby(keys, dropna=False, observed=False)[target].count().reset_index(name="value")
    elif spec.aggregation in {"sum", "mean"}:
        if not pd.api.types.is_numeric_dtype(working[spec.value_column]):
            raise ValueError("합계·평균 집계 대상은 수치형이어야 합니다.")
        grouped = working.groupby(keys, dropna=False, observed=False)[spec.value_column]
        table = (grouped.sum() if spec.aggregation == "sum" else grouped.mean()).reset_index(name="value")
    else:
        table = working.groupby(keys, dropna=False, observed=False).size().reset_index(name="value")
        total = float(table["value"].sum())
        table["value"] = table["value"] / total * 100 if total else 0.0
    return apply_category_orders(table, spec)


def build_chart_statistics(frame, spec):
    spec = spec if isinstance(spec, ChartSpec) else ChartSpec.model_validate(spec)
    for column in [spec.x, spec.y, spec.value_column, *spec.variables]:
        if column and column not in frame.columns:
            raise ValueError(f"데이터에서 변수를 찾을 수 없습니다: {column}")
    if spec.chart_type == "histogram":
        if not pd.api.types.is_numeric_dtype(frame[spec.x]):
            raise ValueError("Histogram X 변수는 수치형이어야 합니다.")
        return frame[[spec.x]].dropna().rename(columns={spec.x: "value"})
    if spec.chart_type == "scatter":
        return frame[[spec.x, spec.y]].dropna().copy()
    if spec.chart_type == "correlation_heatmap":
        numeric = frame[spec.variables].apply(pd.to_numeric, errors="coerce")
        return numeric.corr()
    keys = [spec.x]
    if spec.chart_type in {"grouped_bar", "stacked_bar", "scatter_bubble", "heatmap"}:
        keys.append(spec.y)
    return aggregate_statistics(frame, spec, keys)


def render_chart(frame, spec):
    spec = spec if isinstance(spec, ChartSpec) else ChartSpec.model_validate(spec)
    table = build_chart_statistics(frame, spec)
    fig, ax = plt.subplots(figsize=(10, 6))
    title = spec.title or " · ".join([item for item in (spec.x, spec.y) if item])
    if spec.chart_type == "bar":
        ax.bar(table[spec.x].astype(str), table["value"])
    elif spec.chart_type == "line":
        ax.plot(table[spec.x].astype(str), table["value"], marker="o")
    elif spec.chart_type == "pie":
        ax.pie(table["value"], labels=table[spec.x].astype(str), autopct="%1.1f%%")
    elif spec.chart_type == "histogram":
        ax.hist(table["value"], bins=10, edgecolor="white")
    elif spec.chart_type == "scatter":
        ax.scatter(table[spec.x], table[spec.y], alpha=0.7)
    elif spec.chart_type in {"grouped_bar", "stacked_bar"}:
        pivot = table.pivot_table(index=spec.x, columns=spec.y, values="value", fill_value=0, sort=False)
        pivot.plot(kind="bar", stacked=spec.chart_type == "stacked_bar", ax=ax)
    elif spec.chart_type == "scatter_bubble":
        x_values = pd.Categorical(table[spec.x]).codes
        y_values = pd.Categorical(table[spec.y]).codes
        sizes = 40 + 800 * table["value"] / max(float(table["value"].max()), 1.0)
        ax.scatter(x_values, y_values, s=sizes, alpha=0.6)
        ax.set_xticks(range(table[spec.x].nunique()), pd.unique(table[spec.x]).astype(str))
        ax.set_yticks(range(table[spec.y].nunique()), pd.unique(table[spec.y]).astype(str))
    elif spec.chart_type == "heatmap":
        matrix = table.pivot_table(index=spec.y, columns=spec.x, values="value", fill_value=0, sort=False)
        sns.heatmap(matrix, annot=True, fmt=".2g", cmap="Blues", ax=ax)
    else:
        sns.heatmap(table, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title(title)
    if spec.chart_type not in {"pie", "correlation_heatmap"}:
        if spec.x:
            ax.set_xlabel(spec.x)
        if spec.y and spec.chart_type == "scatter":
            ax.set_ylabel(spec.y)
    if spec.show_values and spec.chart_type in {"bar"}:
        for patch, value in zip(ax.patches, table["value"]):
            ax.annotate(f"{value:,.2g}", (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                        ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    return fig, table


# 전처리 UI
preprocess_column = widgets.Dropdown(description="변수:")
missing_method = widgets.Dropdown(
    description="결측 처리:",
    options=[("특정값", "specific"), ("평균", "mean"), ("중앙값", "median"), ("행 삭제", "drop")],
)
missing_value = widgets.Text(description="처리값:")
btn_missing = widgets.Button(description="결측값 처리", button_style="info")
replace_old = widgets.Text(description="기존값:")
replace_new = widgets.Text(description="변경값:")
btn_replace = widgets.Button(description="특정값 변경", button_style="info")
date_components = widgets.SelectMultiple(
    description="날짜 분리:", options=[("년·월·일", "ymd"), ("월·일", "md"), ("일", "day"), ("시간", "hour")]
)
btn_split_date = widgets.Button(description="날짜 변수 분리")
drop_columns = widgets.SelectMultiple(description="삭제 변수:")
btn_drop = widgets.Button(description="Column 삭제", button_style="danger")
btn_reset_preprocess = widgets.Button(description="전처리 전체 초기화", button_style="warning")
btn_download_clean = widgets.Button(description="전처리 CSV 다운로드", button_style="success")
preprocess_output = widgets.Output()


# 구조화 시각화 UI — 텍스트 요청 컨트롤은 의도적으로 제공하지 않습니다.
chart_type_widget = widgets.Dropdown(description="차트 유형:", options=SUPPORTED_CHARTS)
x_widget = widgets.Dropdown(description="X 변수:")
y_widget = widgets.Dropdown(description="Y 변수:")
value_widget = widgets.Dropdown(description="집계 대상:")
aggregation_widget = widgets.Dropdown(
    description="집계:", options=("count", "valid_count", "sum", "mean", "ratio")
)
variables_widget = widgets.SelectMultiple(description="다중 변수:")
chart_title_widget = widgets.Text(description="제목:")
show_values_widget = widgets.Checkbox(description="값 표시", value=True)
category_order_widget = widgets.Textarea(
    description="범주 순서:", placeholder='예: {"만족도": ["1", "2", "3", "4", "5"]}',
    layout=widgets.Layout(width="100%", height="70px"),
)
btn_make_spec = widgets.Button(description="구조화 메뉴 → ChartSpec", button_style="info")
chart_spec_editor = widgets.Textarea(
    description="ChartSpec:", layout=widgets.Layout(width="100%", height="260px")
)
btn_validate_chart = widgets.Button(description="ChartSpec 검증 및 실행", button_style="success")
visualization_output = widgets.Output()


def refresh_analysis_widgets():
    columns = [] if df_clean is None else [str(column) for column in df_clean.columns]
    optional = [("없음", None)] + [(column, column) for column in columns]
    preprocess_column.options = columns
    drop_columns.options = columns
    x_widget.options = optional
    y_widget.options = optional
    value_widget.options = optional
    variables_widget.options = columns
    if columns:
        preprocess_column.value = columns[0]
        x_widget.value = columns[0]


def show_preprocess_result(message):
    with preprocess_output:
        clear_output(wait=True)
        display(Markdown(message))
        if df_clean is not None:
            display(basic_profile(df_clean))
            display(df_clean.head(5))


def on_missing(_):
    global df_clean
    try:
        frame = require_analysis_data().copy()
        column = preprocess_column.value
        before = int(frame[column].isna().sum())
        method = missing_method.value
        if method == "drop":
            frame = frame.loc[frame[column].notna()].copy()
        elif method in {"mean", "median"}:
            numeric = pd.to_numeric(frame[column], errors="coerce")
            fill = numeric.mean() if method == "mean" else numeric.median()
            frame[column] = frame[column].fillna(fill)
        else:
            frame[column] = frame[column].fillna(coerce_replacement(frame[column], missing_value.value))
        df_clean = frame
        preprocessing_history.append({"operation": "missing", "column": column, "count": before})
        refresh_analysis_widgets()
        show_preprocess_result(f"**결측값 처리 완료:** {column} · {before:,}건")
    except Exception as exc:
        show_preprocess_result(f"**결측값 처리 실패:** {exc}")


def on_replace(_):
    global df_clean
    try:
        frame = require_analysis_data().copy()
        column = preprocess_column.value
        mask = frame[column].astype(str) == replace_old.value
        count = int(mask.sum())
        frame.loc[mask, column] = coerce_replacement(frame[column], replace_new.value)
        df_clean = frame
        preprocessing_history.append({"operation": "replace", "column": column, "count": count})
        show_preprocess_result(f"**특정값 변경 완료:** {column} · {count:,}건")
    except Exception as exc:
        show_preprocess_result(f"**특정값 변경 실패:** {exc}")


def on_split_date(_):
    global df_clean
    try:
        frame = require_analysis_data().copy()
        column = preprocess_column.value
        parsed = pd.to_datetime(frame[column], errors="coerce")
        if parsed.notna().mean() < 0.8:
            raise ValueError("날짜 변환 성공률이 80% 미만입니다.")
        for component in date_components.value:
            if component == "ymd":
                frame[f"{column}_년월일"] = parsed.dt.strftime("%Y년 %m월 %d일")
            elif component == "md":
                frame[f"{column}_월일"] = parsed.dt.strftime("%m월%d일")
            elif component == "day":
                frame[f"{column}_일"] = parsed.dt.strftime("%d일")
            else:
                frame[f"{column}_시간"] = parsed.dt.strftime("%H시")
        df_clean = frame
        preprocessing_history.append({"operation": "split_date", "column": column, "components": list(date_components.value)})
        refresh_analysis_widgets()
        show_preprocess_result("**날짜 파생변수 생성을 완료했습니다.**")
    except Exception as exc:
        show_preprocess_result(f"**날짜 분리 실패:** {exc}")


def on_drop(_):
    global df_clean
    try:
        columns = list(drop_columns.value)
        if not columns:
            raise ValueError("삭제할 변수를 선택하세요.")
        df_clean = require_analysis_data().drop(columns=columns).copy()
        preprocessing_history.append({"operation": "drop_columns", "columns": columns})
        refresh_analysis_widgets()
        show_preprocess_result(f"**Column 삭제 완료:** {', '.join(columns)}")
    except Exception as exc:
        show_preprocess_result(f"**Column 삭제 실패:** {exc}")


def on_reset_preprocess(_):
    global df_clean, preprocessing_history, chart_history, current_chart_spec, current_chart_statistics
    try:
        if df is None:
            raise ValueError("분석 데이터가 없습니다.")
        df_clean = df.copy(deep=True)
        preprocessing_history = []
        chart_history = []
        current_chart_spec = None
        current_chart_statistics = None
        refresh_analysis_widgets()
        show_preprocess_result("**df_clean을 원본 분석 데이터로 초기화했습니다.**")
    except Exception as exc:
        show_preprocess_result(f"**초기화 실패:** {exc}")


def on_download_clean(_):
    try:
        frame = require_analysis_data()
        path = Path("/content/preprocessed_data.csv" if IN_COLAB else tempfile.gettempdir())
        if not IN_COLAB:
            path = path / "preprocessed_data.csv"
        frame.to_csv(path, index=False, encoding="utf-8-sig")
        if IN_COLAB:
            files.download(str(path))
        show_preprocess_result(f"**전처리 데이터를 저장했습니다:** {path}")
    except Exception as exc:
        show_preprocess_result(f"**다운로드 실패:** {exc}")


def menu_chart_spec():
    orders = json.loads(category_order_widget.value) if category_order_widget.value.strip() else {}
    values = {
        "chart_type": chart_type_widget.value,
        "x": x_widget.value,
        "y": y_widget.value,
        "value_column": value_widget.value,
        "aggregation": aggregation_widget.value,
        "title": chart_title_widget.value,
        "show_values": show_values_widget.value,
        "category_orders": orders,
        "variables": list(variables_widget.value),
    }
    return ChartSpec.model_validate(values)


def on_make_spec(_):
    with visualization_output:
        clear_output(wait=True)
        try:
            spec = menu_chart_spec()
            chart_spec_editor.value = json.dumps(spec.model_dump(), ensure_ascii=False, indent=2)
            display(Markdown("**ChartSpec을 생성하고 Pydantic 검증을 완료했습니다.**"))
        except Exception as exc:
            display(Markdown(f"**ChartSpec 생성 실패:** {exc}"))


def on_validate_chart(_):
    global current_chart_spec, current_chart_statistics
    with visualization_output:
        clear_output(wait=True)
        try:
            if not chart_spec_editor.value.strip():
                raise ValueError("먼저 ChartSpec을 생성하거나 입력하세요.")
            raw = json.loads(chart_spec_editor.value)
            spec = ChartSpec.model_validate(raw)
            fig, table = render_chart(require_analysis_data(), spec)
            current_chart_spec = spec.model_dump()
            current_chart_statistics = table.copy()
            chart_history.append({
                "spec": current_chart_spec,
                "statistics": table.head(500).to_dict(orient="records"),
                "source_filename": source_filename,
            })
            display(fig)
            plt.close(fig)
            display(Markdown("### 시각화 통계자료"))
            display(table.head(200))
        except (json.JSONDecodeError, ValidationError, ValueError, TypeError) as exc:
            display(Markdown(f"**검증 실패 · 차트를 실행하지 않았습니다:** {exc}"))


btn_missing.on_click(on_missing)
btn_replace.on_click(on_replace)
btn_split_date.on_click(on_split_date)
btn_drop.on_click(on_drop)
btn_reset_preprocess.on_click(on_reset_preprocess)
btn_download_clean.on_click(on_download_clean)
btn_make_spec.on_click(on_make_spec)
btn_validate_chart.on_click(on_validate_chart)

refresh_analysis_widgets()

preprocess_panel = widgets.VBox([
    widgets.HTML("<h3>df_clean 전처리</h3>"),
    preprocess_column,
    widgets.HBox([missing_method, missing_value, btn_missing]),
    widgets.HBox([replace_old, replace_new, btn_replace]),
    widgets.HBox([date_components, btn_split_date]),
    widgets.HBox([drop_columns, btn_drop]),
    widgets.HBox([btn_reset_preprocess, btn_download_clean]),
    preprocess_output,
])

visualization_panel = widgets.VBox([
    widgets.HTML("<h3>구조화 메뉴 시각화</h3><p>자연어 시각화 요청은 Business Insight Chat에서만 처리합니다.</p>"),
    widgets.HBox([chart_type_widget, aggregation_widget]),
    widgets.HBox([x_widget, y_widget, value_widget]),
    variables_widget,
    widgets.HBox([chart_title_widget, show_values_widget]),
    category_order_widget,
    btn_make_spec,
    chart_spec_editor,
    btn_validate_chart,
    visualization_output,
])


In [5]:
# @title 5. Business Insight 다중 파일 Chat
PROVIDER_MODELS = {
    "OpenAI": ("gpt-5.6-sol", "gpt-5.6-terra", "gpt-5.6-luna"),
    "Gemini": ("gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-pro-preview"),
}
MAX_CONTEXT_CHARS = 80_000
reference_datasets = {}
pending_images = []
app_history = []


class InsightDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")

    action: Literal["text", "chart", "chart_with_insight", "business_insight"]
    dataset_id: str = "main"
    answer: str = ""
    chart_spec: ChartSpec | None = None

    @model_validator(mode="after")
    def validate_decision(self):
        if self.action in {"chart", "chart_with_insight"} and self.chart_spec is None:
            raise ValueError("차트 요청에는 chart_spec이 필요합니다.")
        if self.action in {"text", "business_insight"} and not self.answer.strip():
            raise ValueError("텍스트 요청에는 answer가 필요합니다.")
        return self


AI_SYSTEM_PROMPT = """
당신은 전시회 전문 데이터 분석 수석 컨설턴트입니다.
제공된 데이터셋 요약, 실제 계산 결과, 전처리 이력, 저장된 차트 통계자료와 대화를 근거로 답하세요.
사용자가 답변 언어를 명시하면 그 언어를 최우선으로 따르세요.
데이터에서 확인되지 않은 원인은 가설임을 명시하세요.

[절대 규칙]
- Python, pandas, Matplotlib, Seaborn 코드 또는 코드 블록을 생성하지 마세요.
- 자연어 차트 요청은 지원되는 ChartSpec으로만 변환하세요.
- 지원 차트: bar, line, pie, histogram, scatter, grouped_bar, stacked_bar, scatter_bubble, heatmap, correlation_heatmap.
- ChartSpec에는 실제 데이터셋 컬럼명만 사용하세요.
- 차트 해석은 프로그램이 계산한 실제 통계자료를 받은 후에만 수행하세요.
- 여러 데이터셋을 자동 병합하지 마세요. dataset_id로 사용할 데이터셋 하나를 선택하세요.
- 요청하지 않은 분석이나 권고를 추가하지 마세요.
""".strip()


def decode_text_bytes(raw):
    for encoding in ("utf-8-sig", "cp949", "utf-16", "utf-8"):
        try:
            return raw.decode(encoding)
        except UnicodeError:
            continue
    raise ValueError("텍스트 인코딩을 확인할 수 없습니다.")


def reference_record(filename, raw):
    suffix = Path(filename).suffix.lower()
    record_id = f"ref:{uuid.uuid4().hex}"
    if suffix in {".csv", ".xlsx", ".xls"}:
        frame = read_table_bytes(filename, raw)
        kind = "table"
        text = ""
    elif suffix == ".json":
        data = json.loads(decode_text_bytes(raw))
        if isinstance(data, list) and (not data or isinstance(data[0], dict)):
            frame = pd.DataFrame(data)
            kind = "table"
            text = ""
        else:
            frame = None
            kind = "text"
            text = json.dumps(data, ensure_ascii=False, indent=2)
    elif suffix == ".txt":
        frame = None
        kind = "text"
        text = decode_text_bytes(raw)
    else:
        raise ValueError("Excel, CSV, JSON 또는 TXT 파일만 참고자료로 사용할 수 있습니다.")
    return {
        "id": record_id,
        "filename": Path(filename).name,
        "kind": kind,
        "frame": frame,
        "text": text[:MAX_CONTEXT_CHARS],
    }


def summarize_frame(frame):
    profile = pd.DataFrame({
        "column": [str(c) for c in frame.columns],
        "dtype": [str(frame[c].dtype) for c in frame.columns],
        "valid": [int(frame[c].notna().sum()) for c in frame.columns],
        "missing": [int(frame[c].isna().sum()) for c in frame.columns],
        "unique": [int(frame[c].nunique(dropna=False)) for c in frame.columns],
    })
    try:
        description = frame.describe(include="all").transpose().to_string()
    except Exception:
        description = frame.describe().transpose().to_string()
    return (
        f"크기: {frame.shape}\n컬럼 요약:\n{profile.to_string(index=False)}\n"
        f"기술통계:\n{description}\n상위 5행:\n{frame.head(5).to_string(index=False)}"
    )[:30_000]


def dataset_options():
    options = []
    if df_clean is not None:
        options.append((f"주 분석 데이터 · {source_filename}", "main"))
    for key, record in reference_datasets.items():
        label = f"참고 데이터 · {record['filename']}"
        options.append((label, key))
    return options


def selected_dataset_map():
    result = {}
    for dataset_id in insight_dataset_selector.value:
        if dataset_id == "main" and df_clean is not None:
            result[dataset_id] = {"filename": source_filename, "kind": "table", "frame": df_clean, "text": ""}
        elif dataset_id in reference_datasets:
            result[dataset_id] = reference_datasets[dataset_id]
    return result


def build_insight_context():
    selected = selected_dataset_map()
    sections = []
    for dataset_id, record in selected.items():
        if record["kind"] == "table":
            body = summarize_frame(record["frame"])
        else:
            body = record["text"][:20_000]
        sections.append(f"[dataset_id={dataset_id} · {record['filename']}]\n{body}")
    sections.append("[전처리 이력]\n" + json.dumps(preprocessing_history, ensure_ascii=False, default=str)[:10_000])
    sections.append("[저장된 차트 통계]\n" + json.dumps(chart_history[-5:], ensure_ascii=False, default=str)[:20_000])
    return "\n\n".join(sections)[:MAX_CONTEXT_CHARS]


def history_text():
    rows = []
    for message in app_history[-20:]:
        rows.append(f"{message['role']}: {message.get('text', '')}")
        if message.get("chart_spec"):
            rows.append("ChartSpec: " + json.dumps(message["chart_spec"], ensure_ascii=False))
    return "\n".join(rows)[-20_000:]


def clean_gemini_schema(schema):
    allowed = {
        "$defs", "$ref", "type", "format", "title", "description", "enum", "items",
        "prefixItems", "minItems", "maxItems", "minimum", "maximum", "anyOf", "oneOf",
        "properties", "required",
    }
    def clean(value, parent=None):
        if isinstance(value, list):
            return [clean(item) for item in value]
        if not isinstance(value, dict):
            return value
        if parent in {"properties", "$defs"}:
            return {key: clean(item) for key, item in value.items()}
        return {key: clean(item, key) for key, item in value.items() if key in allowed}
    return clean(schema)


def api_image_parts(provider):
    if provider == "Gemini":
        return [types.Part.from_bytes(data=item["content"], mime_type=item["type"]) for item in pending_images]
    return [
        {
            "type": "input_image",
            "image_url": f"data:{item['type']};base64,{base64.b64encode(item['content']).decode('ascii')}",
        }
        for item in pending_images
    ]


def call_structured_ai(provider, model, prompt):
    key = API_KEYS.get(provider)
    if not key:
        raise ValueError(f"Colab Secrets에 {provider} API 키가 없습니다.")
    if provider == "OpenAI":
        client = OpenAI(api_key=key)
        content = [{"type": "input_text", "text": prompt}] + api_image_parts(provider)
        response = client.responses.parse(
            model=model,
            instructions=AI_SYSTEM_PROMPT,
            input=[{"role": "user", "content": content}],
            text_format=InsightDecision,
        )
        return response.output_parsed
    client = genai.Client(api_key=key)
    parts = [types.Part.from_text(text=prompt)] + api_image_parts(provider)
    response = client.models.generate_content(
        model=model,
        contents=[types.Content(role="user", parts=parts)],
        config=types.GenerateContentConfig(
            system_instruction=AI_SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_json_schema=clean_gemini_schema(InsightDecision.model_json_schema()),
        ),
    )
    return InsightDecision.model_validate_json(response.text)


def call_text_ai(provider, model, prompt):
    key = API_KEYS.get(provider)
    if not key:
        raise ValueError(f"Colab Secrets에 {provider} API 키가 없습니다.")
    if provider == "OpenAI":
        response = OpenAI(api_key=key).responses.create(
            model=model, instructions=AI_SYSTEM_PROMPT, input=prompt
        )
        return (response.output_text or "").strip()
    response = genai.Client(api_key=key).models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=AI_SYSTEM_PROMPT),
    )
    return (response.text or "").strip()


def selected_chart_frame(dataset_id):
    datasets = selected_dataset_map()
    record = datasets.get(dataset_id) or datasets.get("main")
    if not record or record["kind"] != "table":
        raise ValueError("차트에 사용할 표 형식 데이터셋을 선택하세요.")
    return record["frame"], record["filename"]


provider_selector = widgets.Dropdown(description="API Provider:", options=tuple(PROVIDER_MODELS), value=DEFAULT_API_PROVIDER)
model_selector = widgets.Dropdown(description="Model:", options=PROVIDER_MODELS[DEFAULT_API_PROVIDER])
reference_upload = widgets.FileUpload(
    accept=".csv,.xlsx,.xls,.json,.txt", multiple=True, description="참고 파일 업로드"
)
btn_add_references = widgets.Button(description="참고 파일 등록", button_style="info")
insight_dataset_selector = widgets.SelectMultiple(description="사용 데이터셋:", options=[])
image_upload = widgets.FileUpload(accept="image/*", multiple=True, description="이미지 첨부")
history_upload = widgets.FileUpload(accept=".json,.txt,.md", multiple=False, description="기존 대화 등록")
btn_restore_history = widgets.Button(description="기존 대화 복원")
chat_input = widgets.Textarea(
    description="질문:", placeholder="설명, 보고서, 자연어 차트 요청 또는 차트 해석을 입력하세요.",
    layout=widgets.Layout(width="100%", height="90px"),
)
btn_send_chat = widgets.Button(description="질문 전송", button_style="primary")
btn_save_chat = widgets.Button(description="대화 JSON 저장", button_style="success")
btn_new_chat = widgets.Button(description="새 대화", button_style="warning")
insight_status = widgets.HTML()
insight_output = widgets.Output(layout=widgets.Layout(max_height="650px", overflow="auto", border="1px solid #ddd"))


def refresh_insight_dataset_selector():
    options = dataset_options()
    insight_dataset_selector.options = options
    values = [value for _, value in options]
    if "main" in values:
        insight_dataset_selector.value = ("main",)
    elif values:
        insight_dataset_selector.value = (values[0],)


def on_provider_changed(change):
    if change.get("name") == "value":
        model_selector.options = PROVIDER_MODELS[change["new"]]
        model_selector.value = PROVIDER_MODELS[change["new"]][0]


def on_add_references(_):
    successes, failures = [], []
    for item in upload_items(reference_upload.value):
        try:
            record = reference_record(item["name"], item["content"])
            reference_datasets[record["id"]] = record
            successes.append(record["filename"])
        except Exception as exc:
            failures.append(f"{item['name']}: {exc}")
    refresh_insight_dataset_selector()
    message = f"등록 성공 {len(successes)}개"
    if failures:
        message += " · 실패: " + " | ".join(failures)
    insight_status.value = f"<b>{message}</b>"


def on_image_change(change):
    global pending_images
    pending_images = upload_items(change.get("new"))
    insight_status.value = f"이미지 {len(pending_images)}개 첨부 준비"


def render_chat_history():
    with insight_output:
        clear_output(wait=True)
        for message in app_history:
            label = "👤 사용자" if message["role"] == "user" else "🤖 분석가"
            display(Markdown(f"**{label}:**\n\n{message.get('text', '')}"))
            if message.get("chart_spec"):
                if message.get("image_base64"):
                    display(Image(data=base64.b64decode(message["image_base64"]), width=760))
                display(Markdown("**ChartSpec**"))
                display(message["chart_spec"])
                if message.get("statistics"):
                    display(pd.DataFrame(message["statistics"]).head(200))
            display(Markdown("---"))


def on_send_chat(_):
    global pending_images
    question = chat_input.value.strip()
    if not question:
        insight_status.value = "질문을 입력하세요."
        return
    provider, model = provider_selector.value, model_selector.value
    app_history.append({"role": "user", "text": question})
    btn_send_chat.disabled = True
    insight_status.value = f"{provider}가 요청을 분석하고 있습니다..."
    try:
        prompt = f"""[선택 데이터셋과 분석 근거]
{build_insight_context()}

[기존 대화]
{history_text()}

[현재 사용자 요청]
{question}

요청 목적을 분류하고 InsightDecision을 반환하세요. Python 코드는 절대 반환하지 마세요.
"""
        decision = call_structured_ai(provider, model, prompt)
        if decision.action in {"text", "business_insight"}:
            answer = decision.answer
            message = {"role": "model", "text": answer}
        else:
            chart_frame, chart_filename = selected_chart_frame(decision.dataset_id)
            spec = ChartSpec.model_validate(decision.chart_spec.model_dump())
            fig, statistics = render_chart(chart_frame, spec)
            image_buffer = io.BytesIO()
            fig.savefig(image_buffer, format="png", dpi=150, bbox_inches="tight")
            image_base64 = base64.b64encode(image_buffer.getvalue()).decode("ascii")
            plt.close(fig)
            chart_history.append({
                "spec": spec.model_dump(),
                "statistics": statistics.head(500).to_dict(orient="records"),
                "source_filename": chart_filename,
            })
            if decision.action == "chart_with_insight":
                interpretation_prompt = f"[사용자 요청]\n{question}\n\n"
                interpretation_prompt += "[검증된 ChartSpec]\n" + json.dumps(spec.model_dump(), ensure_ascii=False, indent=2)
                interpretation_prompt += "\n\n[실제 계산 통계자료]\n" + statistics.head(500).to_json(force_ascii=False, orient="records")
                interpretation_prompt += "\n\n위 실제 통계자료만 근거로 요청한 언어로 해석하세요."
                answer = call_text_ai(provider, model, interpretation_prompt)
            else:
                answer = f"요청한 차트를 생성했습니다: {spec.title or spec.chart_type}"
            message = {
                "role": "model",
                "text": answer,
                "chart_spec": spec.model_dump(),
                "statistics": statistics.head(500).to_dict(orient="records"),
                "image_base64": image_base64,
            }
        app_history.append(message)
        chat_input.value = ""
        pending_images = []
        render_chat_history()
        insight_status.value = "답변 완료"
    except Exception as exc:
        app_history.pop()  # 실패한 사용자 메시지는 실행 이력에 남기지 않습니다.
        insight_status.value = f"<span style='color:red'>요청 실패: {safe_error_message(exc)}</span>"
    finally:
        btn_send_chat.disabled = False


def on_save_chat(_):
    payload = {
        "schema_version": 3,
        "saved_at": dt.datetime.now(dt.timezone.utc).isoformat(),
        "provider": provider_selector.value,
        "model": model_selector.value,
        "source_filename": source_filename,
        "history": app_history,
    }
    path = Path("/content/exhibition_chat.json" if IN_COLAB else tempfile.gettempdir())
    if not IN_COLAB:
        path = path / "exhibition_chat.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    if IN_COLAB:
        files.download(str(path))
    insight_status.value = f"대화 저장 완료: {path.name}"


def on_restore_history(_):
    global app_history
    try:
        items = upload_items(history_upload.value)
        if not items:
            raise ValueError("복원할 대화 파일을 선택하세요.")
        text = decode_text_bytes(items[0]["content"])
        try:
            payload = json.loads(text)
            restored = payload if isinstance(payload, list) else payload.get("history", [])
        except json.JSONDecodeError:
            restored = [{"role": "user", "text": text, "hidden": True}]
        app_history = [item for item in restored if isinstance(item, dict) and item.get("role") in {"user", "model"}]
        render_chat_history()
        insight_status.value = f"기존 대화 {len(app_history)}개 메시지 복원 완료"
    except Exception as exc:
        insight_status.value = f"<span style='color:red'>대화 복원 실패: {exc}</span>"


def on_new_chat(_):
    global app_history, pending_images
    app_history = []
    pending_images = []
    render_chat_history()
    insight_status.value = "새 대화를 시작했습니다."


provider_selector.observe(on_provider_changed, names="value")
btn_add_references.on_click(on_add_references)
image_upload.observe(on_image_change, names="value")
btn_send_chat.on_click(on_send_chat)
btn_save_chat.on_click(on_save_chat)
btn_restore_history.on_click(on_restore_history)
btn_new_chat.on_click(on_new_chat)

refresh_insight_dataset_selector()

insight_panel = widgets.VBox([
    widgets.HTML("<h3>Business Insight Chat</h3><p>API 키는 Colab Secrets에서만 읽으며 화면에 입력하지 않습니다.</p>"),
    widgets.HBox([provider_selector, model_selector]),
    widgets.HBox([reference_upload, btn_add_references]),
    insight_dataset_selector,
    widgets.HBox([image_upload, history_upload, btn_restore_history]),
    chat_input,
    widgets.HBox([btn_send_chat, btn_save_chat, btn_new_chat]),
    insight_status,
    insight_output,
])


In [6]:
# @title 6. 통합 UI 및 Notebook 자체 테스트
def run_notebook_self_tests():
    global df, df_clean, source_filename, analysis_source, current_chart_spec
    passed = []

    assert select_gemini_key({"GEMINI_API_KEY": "g", "GOOGLE_API_KEY": "o", "exhibition": "e"}) == "g"
    assert select_gemini_key({"GOOGLE_API_KEY": "o", "exhibition": "e"}) == "o"
    assert select_gemini_key({"exhibition": "e"}) == "e"
    passed.append("Colab Secrets 우선순위")

    original_openai_key = API_KEYS.get("OpenAI", "")
    API_KEYS["OpenAI"] = "secret-redaction-test"
    assert "secret-redaction-test" not in safe_error_message("failure: secret-redaction-test")
    API_KEYS["OpenAI"] = original_openai_key
    passed.append("API 비밀값 오류 메시지 마스킹")

    csv_raw = "category,value\nA,10\nB,20\n".encode("utf-8")
    record = dataset_record("sample.csv", csv_raw, "test")
    assert record["rows"] == 2 and record["columns"] == 2
    passed.append("CSV 기본 테스트 파일 읽기")

    accumulated_uploads = (
        {"name": "old.txt", "type": "text/plain", "content": b"old"},
        {"name": "sample.csv", "type": "text/csv", "content": csv_raw},
    )
    selected_upload, ignored_uploads = latest_supported_upload(accumulated_uploads)
    assert selected_upload["name"] == "sample.csv" and ignored_uploads == ["old.txt"]
    passed.append("Colab 누적 업로드에서 최신 유효 파일 선택")

    state_snapshot = {
        "df": df, "df_clean": df_clean, "source_filename": source_filename,
        "analysis_source": analysis_source, "preprocessing_history": list(preprocessing_history),
        "chart_history": list(chart_history), "current_chart_spec": current_chart_spec,
    }
    preprocessing_history.append({"action": "test"})
    chart_history.append({"chart": "test"})
    current_chart_spec = {"chart_type": "bar"}
    replacement_record = dataset_record(
        "replacement.csv", "category,value\nC,30\n".encode("utf-8"), "test"
    )
    set_analysis_dataset(replacement_record, "uploaded")
    assert source_filename == "replacement.csv" and analysis_source == "uploaded"
    assert preprocessing_history == [] and chart_history == [] and current_chart_spec is None
    df = state_snapshot["df"]
    df_clean = state_snapshot["df_clean"]
    source_filename = state_snapshot["source_filename"]
    analysis_source = state_snapshot["analysis_source"]
    preprocessing_history[:] = state_snapshot["preprocessing_history"]
    chart_history[:] = state_snapshot["chart_history"]
    current_chart_spec = state_snapshot["current_chart_spec"]
    refresh_data_status()
    refresh_analysis_widgets()
    refresh_insight_dataset_selector()
    passed.append("분석 원본 선택 및 하위 결과 초기화")

    with tempfile.TemporaryDirectory() as directory:
        save_default_dataset(record, directory)
        loaded = load_default_dataset(directory)
        assert loaded["filename"] == "sample.csv" and loaded["rows"] == 2
        replacement_raw = "category,value\nC,30\n".encode("utf-8")
        replacement = dataset_record("replacement.csv", replacement_raw, "test")
        save_default_dataset(replacement, directory)
        assert load_default_dataset(directory)["filename"] == "replacement.csv"
        delete_default_dataset(directory)
        assert load_default_dataset(directory) is None
    passed.append("기본 파일 저장·자동 불러오기·교체·삭제")

    valid_spec = ChartSpec(
        chart_type="bar", x="category", aggregation="sum", value_column="value",
        category_orders={"category": ["B", "A"]},
    )
    fig, stats = render_chart(record["frame"], valid_spec)
    plt.close(fig)
    assert stats["category"].tolist() == ["B", "A"]
    passed.append("구조화 ChartSpec 생성·통계·차트 실행")

    try:
        ChartSpec(chart_type="heatmap", x="category", aggregation="count")
        raise AssertionError("잘못된 ChartSpec이 허용되었습니다.")
    except ValidationError:
        pass
    passed.append("잘못된 ChartSpec 실행 차단")

    good_reference = reference_record("reference.json", b'[{"group":"A","score":1}]')
    assert good_reference["kind"] == "table"
    failures = []
    for name, raw in (("good.csv", csv_raw), ("bad.bin", b"bad")):
        try:
            reference_record(name, raw)
        except Exception as exc:
            failures.append((name, str(exc)))
    assert len(failures) == 1 and failures[0][0] == "bad.bin"
    passed.append("Business Insight 다중 파일 독립 처리")

    schema_text = json.dumps(InsightDecision.model_json_schema(), ensure_ascii=False)
    assert "python_code" not in schema_text and "chart_spec" in schema_text
    passed.append("Business Insight 자연어 차트는 ChartSpec 전용")

    return passed


SELF_TEST_RESULTS = run_notebook_self_tests()
print(f"Notebook 자체 테스트 완료: {len(SELF_TEST_RESULTS)}개 통과")
for result in SELF_TEST_RESULTS:
    print(" -", result)

tabs = widgets.Tab(children=[data_panel, preprocess_panel, visualization_panel, insight_panel])
for index, title in enumerate(("데이터·기본 현황", "전처리", "구조화 시각화", "Business Insight")):
    tabs.set_title(index, title)
display(tabs)


Notebook 자체 테스트 완료: 10개 통과
 - Colab Secrets 우선순위
 - API 비밀값 오류 메시지 마스킹
 - CSV 기본 테스트 파일 읽기
 - Colab 누적 업로드에서 최신 유효 파일 선택
 - 분석 원본 선택 및 하위 결과 초기화
 - 기본 파일 저장·자동 불러오기·교체·삭제
 - 구조화 ChartSpec 생성·통계·차트 실행
 - 잘못된 ChartSpec 실행 차단
 - Business Insight 다중 파일 독립 처리
 - Business Insight 자연어 차트는 ChartSpec 전용
